# Voronoi Filigree Pendant

A 3D jewelry pendant with an openwork Voronoi lattice pattern, built with:
- **scipy** `SphericalVoronoi` for tessellation on a sphere
- **COMPAS** `Mesh` for mesh construction and manipulation
- **pythreejs** for interactive WebGL rendering with metallic materials

In [1]:
import numpy as np
from scipy.spatial import SphericalVoronoi
from compas.datastructures import Mesh
import pythreejs as p3
from IPython.display import display

N_SEEDS = 90
INSET_FACTOR = 0.28
STRETCH_Y = 1.55
TAPER_STRENGTH = 0.55
SPHERE_RADIUS = 1.0

In [2]:
def fibonacci_sphere(n: int, radius: float = 1.0) -> np.ndarray:
    """Generate approximately-uniform points on a sphere via Fibonacci spiral."""
    golden_ratio = (1 + np.sqrt(5)) / 2
    indices = np.arange(n, dtype=float)
    theta = np.arccos(1 - 2 * (indices + 0.5) / n)
    phi = 2 * np.pi * indices / golden_ratio
    x = radius * np.sin(theta) * np.cos(phi)
    y = radius * np.sin(theta) * np.sin(phi)
    z = radius * np.cos(theta)
    return np.column_stack([x, y, z])

seed_points = fibonacci_sphere(N_SEEDS, SPHERE_RADIUS)
sv = SphericalVoronoi(seed_points, radius=SPHERE_RADIUS, center=np.array([0.0, 0.0, 0.0]))
sv.sort_vertices_of_regions()

print(f"Voronoi regions: {len(sv.regions)}")
print(f"Voronoi vertices: {len(sv.vertices)}")

Voronoi regions: 90
Voronoi vertices: 176


In [3]:
voronoi_vertices = sv.vertices.tolist()
voronoi_faces = [list(region) for region in sv.regions]

voronoi_mesh = Mesh.from_vertices_and_faces(voronoi_vertices, voronoi_faces)

print(f"COMPAS Mesh — vertices: {voronoi_mesh.number_of_vertices()}, "
      f"edges: {voronoi_mesh.number_of_edges()}, "
      f"faces: {voronoi_mesh.number_of_faces()}")

COMPAS Mesh — vertices: 176, edges: 264, faces: 90


In [4]:
def build_filigree_mesh(mesh: Mesh, inset: float) -> tuple[np.ndarray, np.ndarray]:
    """
    Build an openwork lattice from a Voronoi mesh by insetting each face
    and keeping only the edge-strip rings.

    Returns (vertices, triangles) as numpy arrays.
    """
    all_verts: list[list[float]] = []
    all_tris: list[list[int]] = []

    for fkey in mesh.faces():
        face_verts = mesh.face_coordinates(fkey)
        n = len(face_verts)
        if n < 3:
            continue

        pts = np.array(face_verts)
        centroid = pts.mean(axis=0)
        inset_pts = pts + inset * (centroid - pts)

        base_idx = len(all_verts)
        for p in pts:
            all_verts.append(p.tolist())
        for p in inset_pts:
            all_verts.append(p.tolist())

        for i in range(n):
            j = (i + 1) % n
            o0 = base_idx + i
            o1 = base_idx + j
            i0 = base_idx + n + i
            i1 = base_idx + n + j
            all_tris.append([o0, o1, i1])
            all_tris.append([o0, i1, i0])

    return np.array(all_verts, dtype=np.float32), np.array(all_tris, dtype=np.uint32)

lattice_verts, lattice_tris = build_filigree_mesh(voronoi_mesh, INSET_FACTOR)
print(f"Filigree lattice — vertices: {len(lattice_verts)}, triangles: {len(lattice_tris)}")

Filigree lattice — vertices: 1056, triangles: 1056


In [5]:
def deform_to_pendant(vertices: np.ndarray, stretch: float, taper: float) -> np.ndarray:
    """
    Deform a spherical mesh into a teardrop pendant silhouette.
    Z-axis is up: top gets pinched, bottom stays full.
    """
    v = vertices.copy()
    v[:, 2] *= stretch
    z_min, z_max = v[:, 2].min(), v[:, 2].max()
    z_range = z_max - z_min
    if z_range < 1e-8:
        return v
    t = (v[:, 2] - z_min) / z_range
    scale_xz = 1.0 - taper * (t ** 1.4)
    v[:, 0] *= scale_xz
    v[:, 1] *= scale_xz
    return v


def compute_smooth_normals(vertices: np.ndarray, triangles: np.ndarray) -> np.ndarray:
    """Compute per-vertex normals by averaging adjacent face normals."""
    normals = np.zeros_like(vertices, dtype=np.float32)
    v0 = vertices[triangles[:, 0]]
    v1 = vertices[triangles[:, 1]]
    v2 = vertices[triangles[:, 2]]
    face_normals = np.cross(v1 - v0, v2 - v0)
    for i in range(3):
        np.add.at(normals, triangles[:, i], face_normals)
    lengths = np.linalg.norm(normals, axis=1, keepdims=True)
    lengths = np.maximum(lengths, 1e-8)
    normals /= lengths
    return normals


pendant_verts = deform_to_pendant(lattice_verts, STRETCH_Y, TAPER_STRENGTH)
pendant_normals = compute_smooth_normals(pendant_verts, lattice_tris)

print(f"Pendant bounding box:")
print(f"  X: [{pendant_verts[:,0].min():.3f}, {pendant_verts[:,0].max():.3f}]")
print(f"  Y: [{pendant_verts[:,1].min():.3f}, {pendant_verts[:,1].max():.3f}]")
print(f"  Z: [{pendant_verts[:,2].min():.3f}, {pendant_verts[:,2].max():.3f}]")

Pendant bounding box:
  X: [-0.825, 0.816]
  Y: [-0.818, 0.825]
  Z: [-1.533, 1.533]


In [6]:
def build_bail_loop(center_z: float, radius: float = 0.12, tube_r: float = 0.025,
                    n_major: int = 24, n_minor: int = 8) -> tuple[np.ndarray, np.ndarray]:
    """Generate a small torus (bail loop) at the top of the pendant for the chain."""
    verts = []
    tris = []
    for i in range(n_major):
        theta = 2 * np.pi * i / n_major
        for j in range(n_minor):
            phi = 2 * np.pi * j / n_minor
            x = (radius + tube_r * np.cos(phi)) * np.cos(theta)
            y = (radius + tube_r * np.cos(phi)) * np.sin(theta)
            z = tube_r * np.sin(phi) + center_z
            verts.append([x, y, z])

    for i in range(n_major):
        for j in range(n_minor):
            i_next = (i + 1) % n_major
            j_next = (j + 1) % n_minor
            a = i * n_minor + j
            b = i_next * n_minor + j
            c = i_next * n_minor + j_next
            d = i * n_minor + j_next
            tris.append([a, b, c])
            tris.append([a, c, d])

    return np.array(verts, dtype=np.float32), np.array(tris, dtype=np.uint32)


bail_z = pendant_verts[:, 2].max() + 0.08
bail_verts, bail_tris = build_bail_loop(bail_z, radius=0.13, tube_r=0.028)

combined_verts = np.vstack([pendant_verts, bail_verts])
combined_tris = np.vstack([lattice_tris, bail_tris + len(pendant_verts)])
combined_normals = compute_smooth_normals(combined_verts, combined_tris)

print(f"Combined mesh — vertices: {len(combined_verts)}, triangles: {len(combined_tris)}")

Combined mesh — vertices: 1248, triangles: 1440


In [7]:
positions = p3.BufferAttribute(combined_verts, normalized=False)
normals_attr = p3.BufferAttribute(combined_normals, normalized=False)
index = p3.BufferAttribute(combined_tris.ravel().astype(np.uint32), normalized=False)

geom = p3.BufferGeometry(
    attributes={
        "position": positions,
        "normal": normals_attr,
    },
    index=index,
)

gold_material = p3.MeshPhysicalMaterial(
    color="#e8b471",
    metalness=0.88,
    roughness=0.12,
    side="DoubleSide",
    flatShading=False,
    reflectivity=0.9,
)

pendant_mesh = p3.Mesh(geometry=geom, material=gold_material)

edges_geom = p3.EdgesGeometry(geom)
edge_lines = p3.LineSegments(
    geometry=edges_geom,
    material=p3.LineBasicMaterial(color="#8b6914", linewidth=1),
)

ambient = p3.AmbientLight(color="#404040", intensity=0.6)
key_light = p3.DirectionalLight(color="#ffffff", intensity=1.2, position=[5, 5, 5])
fill_light = p3.DirectionalLight(color="#ffeedd", intensity=0.6, position=[-3, 2, -2])
rim_light = p3.DirectionalLight(color="#aaccff", intensity=0.4, position=[0, -4, 3])

camera = p3.PerspectiveCamera(
    position=[0, -3.5, 0.5],
    up=[0, 0, 1],
    fov=40,
    aspect=1.2,
)

scene = p3.Scene(
    children=[pendant_mesh, edge_lines, ambient, key_light, fill_light, rim_light, camera],
    background="#1a1a2e",
)

controls = p3.OrbitControls(controlling=camera, target=[0, 0, 0.3])

renderer = p3.Renderer(
    camera=camera,
    scene=scene,
    controls=[controls],
    width=800,
    height=700,
    antialias=True,
)

display(renderer)

Renderer(camera=PerspectiveCamera(aspect=1.2, fov=40.0, position=(0.0, -3.5, 0.5), projectionMatrix=(1.0, 0.0,…